## Versión final organizada

Esta copia escribe sus salidas bajo `outputs/`. Las rutas comunes se definen en `project_paths.py`.


In [ ]:
import importlib
import project_paths as _project_paths
_project_paths = importlib.reload(_project_paths)
from project_paths import (
    PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR,
    cle_target_experiment_paths, cse_experiment_paths, ensure_output_directories,
)
ensure_output_directories()
EXPERIMENT_NAME = 'xyz_control_nature_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, EXPERIMENT_FIGURES_DIR = cse_experiment_paths(EXPERIMENT_NAME)
CLE_TARGET_EXPERIMENT_ROOT, CLE_TARGET_DATA_DIR, CLE_TARGET_FIGURES_DIR = cle_target_experiment_paths(EXPERIMENT_NAME)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLE_TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLE_TARGET_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# Qiwen CSE xyz control


## Imports


In [ ]:
import importlib
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from class_da_sys import da_system
from class_Lorenz63 import run_RK4
from class_cse import plot_controlled_vs_free_nature_run
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)
from class_cse_qiwen import (
    build_and_apply_perturbation,
    detect_regime_changes,
    find_past_safe_trajectory,
    integrate_and_align,
    lorenz63,
    plot_found_past_safe_trajectory,
    plot_cse_initialization,
    plot_failed_event_diagnostic,
    plot_3d_nature_run_comparison,
    plot_lorenz_attractor_aligned_cse_starts,
    plot_regime_crossings,
    run_single_cse as run_single_cse_core,
    run_single_cse_mini_test,
    summarize_cse_batch_results,
    summarize_cse_batch_results_by_D,
    run_perturbation_unit_test,
)


## DA System Setup


In [ ]:
BASE_PATH = PROJECT_ROOT
PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)

# This notebook assimilates x, y, z and applies CSE perturbations to x, y, and z.
# control_indices=None makes build_and_apply_perturbation skip component masking
# and perturb x, y, and z, which is the original CSE case in Figure 3(a) of the paper.
CONTROL_INDICES = None
CONTROL_LABEL = 'xyz'
REGIME_REFERENCE_MODE = 'nature'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04


## Load OSSE Histories


In [ ]:
with open(OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
# Detect every regime transition in the NATURE trajectory as a sign change
# of the x component (a crossing of x=0 between two consecutive time steps).
# With EVENT_SELECTION='qiwen', the function accepts crossings in both
# directions, keeps those occurring after step 500 whose preceding residence
# time is longer than 385 steps, and retains at most the first 100 events.
# The complete list of crossings is also saved in RESULTS_DIR for reuse.
regime_change_info = detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='cse_qiwen_xyz_control_change_region_time_qiwen_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

# All detected x=0 crossings in the NATURE run.
change_region_time = regime_change_info['change_region_time']
# Forecast/control horizon used by the function's diagnostic sign check.
check_position = regime_change_info['check_position']
# Filtered crossing times used as the CSE event set.
t_start_series = regime_change_info['t_start_series']
# Time spent in the previous regime before each selected crossing.
t_start_residence = regime_change_info['t_start_residence']
# Mean interval between all detected crossings, before event filtering.
mean_crossing_interval = regime_change_info['mean_crossing_interval']


## Visual Check of Regime Crossings


In [ ]:
# Plot the x component of the NATURE trajectory and mark the regime-change
# events selected in the previous cell. The upper panel shows a wide view
# of the first 20,000 steps; the lower panel zooms around the first event.
# Green and orange lines identify the two crossing directions (+ to - and
# - to +), while blue lines mark the DA-cycle-aligned time at which control
# would begin: position steps before each selected crossing.
# The function saves the PNG in PathFigures and returns the Matplotlib
# figure, its two axes, and the complete path of the saved image.
fig_regime_crossings, axes_regime_crossings, regime_crossings_path = plot_regime_crossings(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
    output_filename='regime_crossings_qiwen_qiwenevents.png',
    event_selection=EVENT_SELECTION,
)


## Lorenz Attractor View of Regime-Change Points


In [ ]:
# Show the first 40 DA-cycle-aligned CSE starting points on the Lorenz
# attractor, using one 3D view and the x-y, x-z, and y-z projections.
fig_lorenz_starts, axes_lorenz_starts, lorenz_starts_path = plot_lorenz_attractor_aligned_cse_starts(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
)
lorenz_starts_tagged_path = lorenz_starts_path.with_name(f'{lorenz_starts_path.stem}{OUTPUT_TAG}{lorenz_starts_path.suffix}')
lorenz_starts_path.replace(lorenz_starts_tagged_path)
lorenz_starts_path = lorenz_starts_tagged_path


## CSE Parameters


In [ ]:
# Configure the single-event CSE diagnostic: D is the norm of each xyz
# control perturbation, experiment_step is the number of DA cycles, and
# EVENT_INDEX selects one event from t_start_series.
D = 0.05
amp = 1.0 / D
experiment_step = 1000
EVENT_INDEX = 1

# Lorenz-63 parameters and integration time step used by the CSE.
solver_params = {
    'model_func': lorenz63,
    'sigma': 10.0,
    'beta': 8.0 / 3.0,
    'rho': 28.0,
    'dt': fcst_dt,
}

print('position:', position)


## Initialize CSE Experiment


In [ ]:
# Select one detected regime-change time and place the CSE start position
# steps before it, rounded down to the nearest DA-cycle boundary (Ta).
t_start = t_start_series[EVENT_INDEX]
t0 = (t_start - position) - (t_start - position) % Ta
# Locate that aligned start time in the saved OSSE analysis history.
osse_start_index = (t0 - osse_summary['t0']) // Ta
if osse_summary['t0'] + osse_start_index * Ta != t0:
    raise ValueError('t0 is not aligned with the OSSE grid')

# Initialize the analysis ensemble and the controlled truth from the same
# absolute time t0.
Xa_initial = np.matrix(osse_Xa_history[osse_start_index])  # (xdim, edim)
x_original = np.asarray(x_nature[:, t0]).ravel().copy()

# Allocate storage for the analysis ensemble at every DA cycle and for the
# controlled truth at every model time step.
Xa_history_control = np.zeros((experiment_step + 1, das.xdim, das.edim))
Xa_history_control[0] = osse_Xa_history[osse_start_index]
evo_history = np.zeros((3, experiment_step * Ta + 1))
evo_history[:, 0] = x_original

# Check the ensemble dimensions and measure its initial mean-state error
# relative to the NATURE state.
assert Xa_initial.shape == (das.xdim, das.edim)
rmse_init = np.linalg.norm(np.asarray(Xa_initial).mean(axis=1) - x_original) / np.sqrt(3)

print('rmse_init:', rmse_init)
print('t0:', t0)
print('osse_start_index:', osse_start_index)
print('np.sign(x_original[0]):', np.sign(x_original[0]))


## Visual Check of Initialization


In [ ]:
# Pre-run visual check of the CSE initialization.
# This confirms that t0 is correctly placed before the selected event and
# shows where the inherited OSSE analysis ensemble sits relative to nature.
fig_cse_init, ax_cse_init, cse_init_path = plot_cse_initialization(
    x_nature=x_nature,
    Xa_initial=Xa_initial,
    t0=t0,
    t_start=t_start,
    osse_start_index=osse_start_index,
    rmse_init=rmse_init,
    path_figures=PathFigures,
    output_filename='cse_initialization_qiwen_qiwenevents.png',
)


## Run Single CSE Event

Ejecuta el evento seleccionado y exporta `cse_target.pkl`, `cse_interventions.pkl` y `cse_summary.pkl` para el posterior cálculo de `CLE_TARGET`.


In [ ]:
# Reload the edited module so this cell can be rerun without restarting the kernel.
import importlib
import class_cse_qiwen as _cse_qiwen_module
_cse_qiwen_module = importlib.reload(_cse_qiwen_module)
run_single_cse_core = _cse_qiwen_module.run_single_cse

# Connect the notebook configuration to the authoritative CSE implementation.
# The result dictionary is returned unchanged, including all diagnostic histories.
def run_single_cse(event_index, D_run, seed, regime_reference='nature', candidate_pool='full', cle_target_output_dir=None):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_number,
        cle_target_output_dir=cle_target_output_dir,
    )


# Run the selected event with the parameters defined above.
single_cse_result = run_single_cse(
    event_index=EVENT_INDEX,
    D_run=D,
    seed=1,
    regime_reference='nature',
    candidate_pool='full',
    cle_target_output_dir=CLE_TARGET_DATA_DIR,
)

print('Event index:', single_cse_result['event_index'])
print('Success:', single_cse_result['success'])
print('Failure cycle:', single_cse_result['failure_cycle'])
print('Completed cycles:', single_cse_result['n_completed'])
print('Control ratio:', single_cse_result['control_ratio'])
print('False alarm ratio:', single_cse_result['false_alarm_ratio'])
print('Success control ratio:', single_cse_result['success_control_ratio'])
print('Branch counts:', single_cse_result['branch_counts'])
print('Safe searches:', single_cse_result['n_safe_searches'])
print('CLE_TARGET input files:')
for cle_input_name, cle_input_path in single_cse_result['cle_target_files'].items():
    print(f'  {cle_input_name}: {cle_input_path}')

assert single_cse_result['success'], 'run_single_cse did not achieve control.'


## Plot Controlled NR vs Free NR


In [ ]:
# Reload the plotting module so this cell also sees edits without restarting the kernel.
import importlib
import class_cse as _class_cse_module
_class_cse_module = importlib.reload(_class_cse_module)
plot_controlled_vs_free_nature_run = _class_cse_module.plot_controlled_vs_free_nature_run

# Plot the reusable run_single_cse result against the corresponding free NATURE trajectory.
fig, axes, controlled_vs_free_path = plot_controlled_vs_free_nature_run(
    evo_history=single_cse_result['evo_history'],
    x_nature=x_nature,
    t0=single_cse_result['t0'],
    n_completed=single_cse_result['n_completed'],
    Ta=Ta,
    y_obs=y_obs,
    yo_history=single_cse_result['yo_history'],
    branch_history=single_cse_result['branch_history'],
    perturbed_time=single_cse_result['perturbed_time'],
    xdim=das.xdim,
    path_figures=PathFigures,
    output_filename='nr_controlled_vs_nr_qiwenevents.png',
)


## 3D Plot Controlled NR vs Free NR


In [ ]:
# Plot the current run_single_cse result, with the same saved-result fallback.
import importlib
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)

PathFigures = globals().get('PathFigures', EXPERIMENT_FIGURES_DIR)
plot_3d_data = cse_qiwen.plot_3d_nature_run_comparison_with_fallback(
    current_result=globals().get('single_cse_result'),
    x_nature=globals().get('x_nature'),
    Ta=globals().get('Ta'),
    path_figures=PathFigures,
    analysis_data_path=ANALYSIS_DATA_DIR / 'x_analysis_init.pkl',
    results_dir=RESULTS_DIR,
    output_tag=OUTPUT_TAG,
    D=globals().get('SUMMARY_D', globals().get('D', 0.05)),
    event_index=globals().get('EVENT_INDEX', 0),
    experiment_step=globals().get('experiment_step'),
    output_filename='NR_3D_comparison_after_control_qiwenevents.png',
)

plot_result = plot_3d_data['plot_result']
plot_evo_history = plot_3d_data['plot_evo_history']
plot_t0 = plot_3d_data['plot_t0']
plot_num_cycles = plot_3d_data['plot_num_cycles']
x_nature = plot_3d_data['x_nature']
Ta = plot_3d_data['Ta']
x_nature_nocontrol_3d = plot_3d_data['nature_nocontrol']
x_nature_control_3d = plot_3d_data['nature_control']
fig_3d_nr = plot_3d_data['fig']
axes_3d_nr = plot_3d_data['axes']
nr_3d_path = plot_3d_data['output_path']
